---
### Problem 1 — Imports and Data Loading

নিচের সব library import করো:
- `numpy`, `pandas`
- `sklearn` থেকে: `train_test_split`, `SimpleImputer`, `OrdinalEncoder`, `OneHotEncoder`, `LabelEncoder`, `StandardScaler`, `MinMaxScaler`, `Pipeline`, `ColumnTransformer`, `LogisticRegression`

`titanic_data_updated.csv` লোড করো `df` নামে এবং প্রথম 5টি row দেখাও।

In [19]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression

In [20]:
df = pd.read_csv("titanic_data_updated.csv")
df.head(10)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,no,third,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,yes,first,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,yes,third,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,yes,first,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,no,third,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S
5,6,no,third,"Moran, Mr. James",male,NaN,0,0,330877,8.4583,NaN,Q
6,7,no,first,"McCarthy, Mr. Timothy J",male,54.0,0,0,17463,51.8625,E46,S
7,8,no,third,"Palsson, Master. Gosta Leonard",male,2.0,3,1,349909,21.0750,NaN,S
8,9,yes,third,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",female,27.0,0,2,347742,11.1333,NaN,S
9,10,yes,second,"Nasser, Mrs. Nicholas (Adele Achem)",female,14.0,1,0,237736,30.0708,NaN,C


---
### Problem 2 — Feature Engineering

দুটি নতুন column তৈরি করো:

1. `Family_Size` — formula: `SibSp + Parch + 1`
2. `Deck` — `Cabin` column-এর NaN গুলো `"Missing"` দিয়ে fill করো, তারপর প্রথম character নিয়ে `Deck` column তৈরি করো

শেষে `df.sample(5)` দিয়ে result দেখাও।

In [21]:
df['Family_Size'] = df['SibSp'] + df['Parch'] + 1
df['Cabin'] = df['Cabin'].fillna("Missing")

df['Deck'] = df['Cabin'].astype(str).str[0]
df.sample(5)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked,Family_Size,Deck
625,626,no,first,"Sutton, Mr. Frederick",male,61.0,0,0,36963,32.3208,D50,S,1,D
566,567,no,third,"Stoytcheff, Mr. Ilia",male,19.0,0,0,349205,7.8958,Missing,S,1,M
283,284,yes,third,"Dorking, Mr. Edward Arthur",male,19.0,0,0,A/5. 10482,8.0500,Missing,S,1,M
302,303,no,third,"Johnson, Mr. William Cahoone Jr",male,19.0,0,0,LINE,0.0000,Missing,S,1,M
528,529,no,third,"Salonen, Mr. Johan Werner",male,39.0,0,0,3101296,7.9250,Missing,S,1,M


---
### Problem 3 — X and y Split

`df` থেকে features এবং target আলাদা করো:

- `X` = `Survived` বাদে সব column
- `y` = শুধু `Survived` column

`X.shape` এবং `y.shape` print করো।

In [22]:
X = df.drop('Survived', axis=1)
y = df['Survived']

In [23]:
X.shape

(891, 13)

In [24]:
y.shape

(891,)

---
### Problem 4 — Train-Test Split

`train_test_split` ব্যবহার করে `X_train`, `X_test`, `y_train`, `y_test` তৈরি করো।

Parameters:
- `test_size = 0.2`
- `random_state = 42`
- `stratify = y`

`X_train` এবং `X_test`-এর shape print করো।

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42, stratify = y)

In [26]:
X_train.shape

(712, 13)

In [27]:
X_test.shape

(179, 13)

---
### Problem 5 — Outlier Handling

**Age column — Z-score method (rows remove করো):**
- `Z_score = (Age - mean) / std`
- যেসব row-এ `|Z_score| > 3`, সেগুলো `X_train` এবং `y_train` থেকে বাদ দাও

**Fare column — IQR clipping (rows রাখো, values clip করো):**
- `Q1` = 25th percentile, `Q3` = 75th percentile
- `IQR = Q3 - Q1`
- `minimum = max(0, Q1 - 1.5 * IQR)`
- `maximum = Q3 + 1.5 * IQR`
- `clip()` দিয়ে Fare column-কে `[minimum, maximum]` range-এ রাখো

In [28]:
# age 
mean_age = X_train['Age'].mean()
std_age = X_train['Age'].std()

X_train['Z_score'] = (X_train['Age'] - mean_age) / std_age

musk = (abs(X_train['Z_score']) <= 3)

X_train = X_train[musk]
y_train = y_train[musk]

#fare
fare_q1 = X_train['Fare'].quantile(0.25)
fare_q3 = X_train['Fare'].quantile(0.75)

IQR = fare_q3 - fare_q1

minimum = max(0, fare_q1-1.5*IQR)
maximum = fare_q3+1.5*IQR

X_train['Fare'] = X_train['Fare'].clip(minimum, maximum)

---
### Problem 6 — Numerical Pipelines

দুটি numerical pipeline তৈরি করো:

**p1** — `Age` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='mean')`
- Step 2: `StandardScaler()`

**p2** — `Fare` এবং `Family_Size` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='median')`
- Step 2: `MinMaxScaler()`

In [29]:
# numerical 

p1 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='mean')),
        ('scaler', StandardScaler())
    ]
)

p2 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', MinMaxScaler())
    ]
)



---
### Problem 7 — Categorical Pipelines and ColumnTransformer

দুটি categorical pipeline তৈরি করো:

**p3** — `Embarked`, `Sex`, `Deck` column-এর জন্য:
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore')`

**p4** — `Pclass` column-এর জন্য:
- `categories = [['third', 'second', 'first']]`
- Step 1: `SimpleImputer(strategy='most_frequent')`
- Step 2: `OrdinalEncoder(categories=categories)`
- Step 3: `MinMaxScaler()`

তারপর `ColumnTransformer` দিয়ে `preprocessor` তৈরি করো:
- `pipeline_1` → `p1` → `['Age']`
- `pipeline_2` → `p2` → `['Fare', 'Family_Size']`
- `pipeline_3` → `p3` → `['Embarked', 'Sex', 'Deck']`
- `pipeline_4` → `p4` → `['Pclass']`
- `remainder='drop'`

In [30]:
categories = [['third', 'second', 'first']]

In [31]:
# categorical

p3 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(sparse_output=False, drop='first', handle_unknown='ignore'))
    ]
)

p4 = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OrdinalEncoder(categories=categories)),
        ('scaler', MinMaxScaler())
    ]
)


In [32]:
preprocessor = ColumnTransformer(
    transformers=[
        ('pipeline_1', p1,['Age']),
        ('pipeline_2', p2,['Fare', 'Family_Size']),
        ('pipeline_3', p3,['Embarked', 'Sex', 'Deck']),
        ('pipeline_4', p4,['Pclass'])
        
    ],
    remainder='drop'

    
)

preprocessor

ColumnTransformer(transformers=[('pipeline_1',
                                 Pipeline(steps=[('imputer', SimpleImputer()),
                                                 ('scaler', StandardScaler())]),
                                 ['Age']),
                                ('pipeline_2',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Fare', 'Family_Size']),
                                ('pipeline_3',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(drop='first',
                                                                handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Embarked', 'Sex', 'Deck']),
                                ('pipeline_4',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OrdinalEncoder(categories=[['third',
                                                                              'second',
                                                                              'first']])),
                                                 ('scaler', MinMaxScaler())]),
                                 ['Pclass'])])

---
### Problem 8 — Target Column Encoding

`y_train` এবং `y_test`-এ এখন `"yes"` এবং `"no"` string আছে। Model train করার আগে এগুলো numeric করতে হবে।

- `LabelEncoder` দিয়ে `y_train` এবং `y_test` encode করো
- encode করার আগে এবং পরে unique values print করো

In [33]:
le = LabelEncoder()

le.fit(y_train)

y_train = le.transform(y_train)
y_test = le.transform(y_test)